In [1]:
from pathlib import Path
import pandas as pd

RANDOM_STATE = 42

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)
DATA_PATH = PROJECT_ROOT / "data" / "creditcard.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)

print(f"Original shape: {df_raw.shape}")
print(f"Exact duplicate rows: {df_raw.duplicated().sum():,}")

# Keep the first occurrence of each exact duplicate row
df_clean = df_raw.drop_duplicates().copy()

print(f"Shape after deduplication: {df_clean.shape}")
print(f"Remaining exact duplicates: {df_clean.duplicated().sum():,}")
print("\nTarget distribution after deduplication:")
display(
    df_clean["Class"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Legitimate", 1: "Fraudulent"})
    .to_frame("Count")
)

Original shape: (284807, 31)
Exact duplicate rows: 1,081
Shape after deduplication: (283726, 31)
Remaining exact duplicates: 0

Target distribution after deduplication:


,Count
Class,
Legitimate,283253
Fraudulent,473


In [2]:
# Separate predictors and target
X = df_clean.drop(columns=["Class"]).copy()
y = df_clean["Class"].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {X.columns.tolist()}")
print(f"\nFraud prevalence: {y.mean() * 100:.4f}%")
print("\nTarget counts:")
display(
    y.value_counts()
    .sort_index()
    .rename(index={0: "Legitimate", 1: "Fraudulent"})
    .to_frame("Count")
)

Feature matrix shape: (283726, 30)
Target shape: (283726,)
Feature columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

Fraud prevalence: 0.1667%

Target counts:


,Count
Class,
Legitimate,283253
Fraudulent,473


In [3]:
from sklearn.model_selection import train_test_split

# First, reserve 15% for the final test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Split the remaining 85% into training and validation.
# 0.15 / 0.85 = approximately 17.647% of the remaining data.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.15 / 0.85,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

# Verify split sizes and fraud counts
split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Rows": [len(X_train), len(X_val), len(X_test)],
    "Fraud Cases": [
        int(y_train.sum()),
        int(y_val.sum()),
        int(y_test.sum()),
    ],
    "Fraud Prevalence (%)": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100,
    ],
})

display(split_summary.round(4))

print(f"Total rows across splits: {len(X_train) + len(X_val) + len(X_test):,}")
print(f"Original cleaned rows: {len(X):,}")

,Split,Rows,Fraud Cases,Fraud Prevalence (%)
0,Train,198608,331,0.1667
1,Validation,42559,71,0.1668
2,Test,42559,71,0.1668


Total rows across splits: 283,726
Original cleaned rows: 283,726


In [4]:
from sklearn.preprocessing import RobustScaler

scale_columns = ["Time", "Amount"]

# Keep copies so the original split data stays unchanged
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# Fit only on training data
scaler = RobustScaler()
X_train_scaled[scale_columns] = scaler.fit_transform(
    X_train[scale_columns]
)

# Apply the same fitted scaler to validation and test
X_val_scaled[scale_columns] = scaler.transform(
    X_val[scale_columns]
)
X_test_scaled[scale_columns] = scaler.transform(
    X_test[scale_columns]
)

print("RobustScaler fitted on training data only.")
print("Scaled columns:", scale_columns)

display(X_train_scaled[scale_columns].describe().round(4))

RobustScaler fitted on training data only.
Scaled columns: ['Time', 'Amount']


,Time,Amount
count,198608.0000,198608.0000
mean,0.1194,0.9179
std,0.5576,3.3585
min,-0.9947,-0.3057
25%,-0.3584,-0.2269
50%,0.0000,0.0000
75%,0.6416,0.7731
max,1.0352,272.0492


In [6]:
import numpy as np

# Verify transformed data integrity
for split_name, X_split in [
    ("Train", X_train_scaled),
    ("Validation", X_val_scaled),
    ("Test", X_test_scaled),
]:
    print(
        f"{split_name}: shape={X_split.shape}, "
        f"all finite={np.isfinite(X_split.to_numpy()).all()}, "
        f"missing values={X_split.isna().sum().sum()}"
    )

print("\nFeature columns preserved:",
      X_train_scaled.columns.equals(X.columns))

print("Train/validation/test row counts:",
      len(X_train_scaled), len(X_val_scaled), len(X_test_scaled))

Train: shape=(198608, 30), all finite=True, missing values=0
Validation: shape=(42559, 30), all finite=True, missing values=0
Test: shape=(42559, 30), all finite=True, missing values=0

Feature columns preserved: True
Train/validation/test row counts: 198608 42559 42559


In [7]:

from pathlib import Path

# Save processed splits without modifying the original dataset
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

splits = {
    "train": X_train_scaled.assign(Class=y_train),
    "validation": X_val_scaled.assign(Class=y_val),
    "test": X_test_scaled.assign(Class=y_test),
}

for split_name, split_df in splits.items():
    output_path = ARTIFACTS_DIR / f"{split_name}_processed.parquet"
    split_df.to_parquet(output_path, index=False)
    print(f"Saved {split_name}: {split_df.shape} -> {output_path}")

print("All processed splits saved.")


Saved train: (198608, 31) -> c:\Users\Administrator\Self Study\fraud_detection\fraud-detection\fraud-detection\artifacts\train_processed.parquet
Saved validation: (42559, 31) -> c:\Users\Administrator\Self Study\fraud_detection\fraud-detection\fraud-detection\artifacts\validation_processed.parquet
Saved test: (42559, 31) -> c:\Users\Administrator\Self Study\fraud_detection\fraud-detection\fraud-detection\artifacts\test_processed.parquet
All processed splits saved.


In [8]:

import joblib

scaler_path = ARTIFACTS_DIR / "robust_scaler.joblib"

joblib.dump(scaler, scaler_path)

print("Saved fitted scaler:", scaler_path)
print("Scaler type:", type(scaler).__name__)
print("Features scaled:", list(scaler.feature_names_in_))


Saved fitted scaler: c:\Users\Administrator\Self Study\fraud_detection\fraud-detection\fraud-detection\artifacts\robust_scaler.joblib
Scaler type: RobustScaler
Features scaled: ['Time', 'Amount']
